<a href="https://colab.research.google.com/github/joiepark/assignment_esaa/blob/main/260913_%ED%85%8D%EC%8A%A4%ED%8A%B8%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 문서 군집화 소개와 Opinion Review 데이터 세트 실습
### 문서 군집화 개념

*   문서 군집화(Document Clustering)는 비슷한 텍스트 구성의 문서를 군집화(Clustering)하는 것임. 문서 군집화는 동일한 군집에 속하는 문서를 같은 카테고리 소속으로 분류할 수 있으므로 텍스트 분류 기반의 문서 분류와 유사함. 하지만 **텍스트 분류 기반의 문서 분류는 사전에 결정 카테고리 값을 가진 학습 데이터 세트가 필요한 데 반해, 문서 군집화는 학습 데이터 세트가 필요 없는 비지도학습 기반으로 동작함.**

## Opinion Review 데이터 세트를 이용한 문서 군집화 수행
*   해당 데이터 세트는 51개의 텍스트 파일로 구성되어 있으며, 각 파일은 Tripadvisor(호텔), Edmunds.com(자동차), Amazon.com(전자제품) 사이트에서 가져온 리뷰 문서임. 각 문서는 약 100개 정도의 문장을 가지고 있음. 문서 군집화를 이용해 각 리뷰를 분류해보고자 함.



In [23]:
# 디렉터리 내에 있는 파일을 하나씩 읽어서 파일명과 파일 리뷰를 하나의 DataFrame으로 로드

import pandas as pd
import glob, os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
path = r'/content/drive/MyDrive/OpinosisDataset1.0/topics'
all_files = glob.glob(os.path.join(path, "*.data"))
filename_list = []
opinion_text = []

for file_ in all_files:
  df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
  filename_ = file_.split('\\')[-1]     ## 절대경로로 주어진 파일명을 가공
  filename = filename_.split('.')[0]     ## 맨 마지막 .data 확장자도 제거
  filename_list.append(filename)
  opinion_text.append(df.to_string())

document_df = pd.DataFrame({'filename':filename_list, 'opinion_text':opinion_text})
document_df.head()

,filename,opinion_text
0,/content/drive/MyDrive/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...
1,/content/drive/MyDrive/OpinosisDataset1,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...
2,/content/drive/MyDrive/OpinosisDataset1,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t..."
3,/content/drive/MyDrive/OpinosisDataset1,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k..."
4,/content/drive/MyDrive/OpinosisDataset1,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...




*   TfidfVectorizer는 Lemmatization 같은 어근 변환을 직접 지원하진 않지만, tokenizer 인자에 커스텀 어근 변환 함수를 적용해 어근 변환을 수행할 수 있음.



In [25]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

def LemNormalize(text):
    Lemmatizer = WordNetLemmatizer()
    return [Lemmatizer.lemmatize(token) for token in word_tokenize(text)]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [26]:
# 문서를 TF-IDF 형태로 피처 벡터화

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english', ngram_range=(1,2), min_df=0.05, max_df=0.85)
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

In [27]:
# 군집화를 수행해 어떤 문서끼리 군집되는지 확인

from sklearn.cluster import KMeans

## 5개 지합으로 군집화 수행
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_

In [28]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,4
1,/content/drive/MyDrive/OpinosisDataset1,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,4
2,/content/drive/MyDrive/OpinosisDataset1,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t...",4
3,/content/drive/MyDrive/OpinosisDataset1,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",4
4,/content/drive/MyDrive/OpinosisDataset1,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...,4




*   판다스 DataFrame의 sort_valuess(by=정렬칼럼명)를 수행하면 인자로 입력된 '정렬칼럼명'으로 데이터를 정렬할 수 있음.



In [29]:
# 군집화 결과 확인 (cluster_label=0인 데이터 세트)
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
20,/content/drive/MyDrive/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",0
27,/content/drive/MyDrive/OpinosisDataset1,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",0
36,/content/drive/MyDrive/OpinosisDataset1,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,0
38,/content/drive/MyDrive/OpinosisDataset1,"Front seats are very uncomfortable .\n0 No memory seats, no trip computer, can only display outside temp with trip odometer .\n1 ...",0
39,/content/drive/MyDrive/OpinosisDataset1,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,0
40,/content/drive/MyDrive/OpinosisDataset1,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",0
41,/content/drive/MyDrive/OpinosisDataset1,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",0
42,/content/drive/MyDrive/OpinosisDataset1,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",0
43,/content/drive/MyDrive/OpinosisDataset1,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,0
48,/content/drive/MyDrive/OpinosisDataset1,"After slowing down, transmission has to be kicked to speed up .\n0 ...",0


In [31]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
18,/content/drive/MyDrive/OpinosisDataset1,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",1
21,/content/drive/MyDrive/OpinosisDataset1,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",1
35,/content/drive/MyDrive/OpinosisDataset1,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",1
47,/content/drive/MyDrive/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",1
49,/content/drive/MyDrive/OpinosisDataset1,"Great Location , Nice Rooms , H...",1


In [32]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
23,/content/drive/MyDrive/OpinosisDataset1,The food for our event was delicious .\n0 ...,2
30,/content/drive/MyDrive/OpinosisDataset1,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,2


In [33]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
19,/content/drive/MyDrive/OpinosisDataset1,"The staff at Swissotel were not particularly nice .\n0 Each time I waited at the counter for staff for several minutes and then was waved to the desk upon my turn with no hello or anything, or apology for waiting in line .\n1 ...",3
25,/content/drive/MyDrive/OpinosisDataset1,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",3
28,/content/drive/MyDrive/OpinosisDataset1,Mediocre room and service for a very extravagant price .\n0 ...,3
33,/content/drive/MyDrive/OpinosisDataset1,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",3
37,/content/drive/MyDrive/OpinosisDataset1,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,3
44,/content/drive/MyDrive/OpinosisDataset1,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",3
45,/content/drive/MyDrive/OpinosisDataset1,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",3
46,/content/drive/MyDrive/OpinosisDataset1,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,3
50,/content/drive/MyDrive/OpinosisDataset1,Staff are friendl...,3


In [34]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,4
31,/content/drive/MyDrive/OpinosisDataset1,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",4
29,/content/drive/MyDrive/OpinosisDataset1,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",4
26,/content/drive/MyDrive/OpinosisDataset1,"The Eee Super Hybrid Engine utility lets users overclock or underclock their Eee PC's to boost performance or provide better battery life depending on their immediate requirements .\n0 In Super Performance mode CPU, Z shows the bus speed to increase up to 169 .\n1 One...",4
24,/content/drive/MyDrive/OpinosisDataset1,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",4
22,/content/drive/MyDrive/OpinosisDataset1,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",4
17,/content/drive/MyDrive/OpinosisDataset1,"It's fast to acquire satellites .\n0 If you've ever had a Brand X GPS take you on some strange route that adds 20 minutes to your trip, has you turn the wrong way down a one way road, tell you to turn AFTER you've passed the street, frequently loses the satellite signal, or has old maps missing streets, you know how important this stuff is .\n1 ...",4
16,/content/drive/MyDrive/OpinosisDataset1,It is easy to read and when touching the screen it works great !\n0 and zoom out buttons on the 255w to the same side of the screen which makes it a bit easier .\n1 ...,4
15,/content/drive/MyDrive/OpinosisDataset1,"Windows 7 is quite simply faster, more stable, boots faster, goes to sleep faster, comes back from sleep faster, manages your files better and on top of that it's beautiful to look at and easy to use .\n0 , faster about 20% to 30% faster at running applications than my Vista , seriously\n1 ...",4
14,/content/drive/MyDrive/OpinosisDataset1,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,4


전반적으로 군집화된 결과를 살펴보면 군집 개수가 약간 많게 설정되어 있어 세분화되어 군집화된 경향이 있음.

In [35]:
# 중심 개수를 5개에서 3개로 낮춰 3개 그룹으로 군집화한 뒤 결과 확인
from sklearn.cluster import KMeans

km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
7,/content/drive/MyDrive/OpinosisDataset1,headphone jack i got a clear case for it and it i got a clear case for it and it like prvents me from being able to put the jack all the way in so the sound can b messsed up or i can get it in there and its playing well them go to move or something and it slides out .\n0 Picture and sound quality are excellent for this typ of devic .\n1 ...,0
4,/content/drive/MyDrive/OpinosisDataset1,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...,0
9,/content/drive/MyDrive/OpinosisDataset1,"I bought the 8, gig Ipod Nano that has the built, in video camera .\n0 Itunes has an on, line store, where you may purchase and download music and videos which will install onto the ipod .\n1 ...",0
8,/content/drive/MyDrive/OpinosisDataset1,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",0
13,/content/drive/MyDrive/OpinosisDataset1,"I had to uninstall anti, virus and selected other programs, some of which did not have listings in the Programs and Features Control Panel section .\n0 This review briefly touches upon some of the key features and enhancements of Microsoft's latest OS .\n1 ...",0
27,/content/drive/MyDrive/OpinosisDataset1,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",0
29,/content/drive/MyDrive/OpinosisDataset1,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",0
20,/content/drive/MyDrive/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",0
24,/content/drive/MyDrive/OpinosisDataset1,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",0
41,/content/drive/MyDrive/OpinosisDataset1,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",0


### 군집별 핵심 단어 추출하기


*   **각 군집에 속한 문서는 핵심 단어를 주축으로 군집화**되어 있을 것임.
*   **KMeans 객체는 각 군집을 구성하는 단어 피처가 군집의 중심(Centroid)을 기준으로 얼마나 가깝게 위치해 있는지 cluster_centers_라는 속성**으로 제공함. cluster_centers_는 **배열 값으로 제공되며, 행은 개별 군집을, 열은 개별 피처**를 의미함. 각 배열 내의 값은 개별 군집 내의 상대 위치를 숫자 값으로 표현한 일종의 좌표 값임. (e.g. cluster_centers[0, 1]은 0번 군집에서 두 번째 피처 위치 값임.)



In [36]:
# 바로 앞 예제에서 군집 3개로 생성한 KMeans 객체에서 cluster_centers_ 속성값을 가져온 후 값을 확인

cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape: ', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape:  (3, 6155)
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.0015685  0.00073781 0.00055628 ... 0.00067576 0.00010162 0.00010162]
 [0.         0.00216956 0.         ... 0.00131462 0.0024508  0.0024508 ]]


이는 군집이 3개, word 피처가 6155개로 구성되었음을 의미함.


*   각 행의 배열 값은 각 군집 내의 피처의 위치가 개별 중심과 얼마나 가까운가를 상대 값으로 나타낸 것임. 0부터 1까지의 값을 가질 수 있으며 1에 가까울수록 중심과 가까운 값을 의미함.
*   cluster_centers_ 속성값을 이용해 각 군집별 핵심 단어를 찾을 때, **cluster_centers_ 속성은 넘파이의 ndarray이므로 ndarray의 argsort()[:,::-1]을 이용하면 cluster_centers 배열 내 값이 큰 순으로 정렬된 위치 인덱스 값을 반환함**. (* 큰 값으로 정렬한 값을 반환하는 게 아니라 **큰 값을 가진 배열 내 위치 인덱스 값을 반환함.**) 이 위치 인덱스 값이 필요한 이유는 핵심 단어 피처의 이름을 출력하기 위함임.



In [37]:
# get_cluster_details()를 정의

def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10):
  cluster_details = {}
  ## cluster_centers_ 배열 내에서 가장 값이 큰 데이터의 위치 인덱스를 추출한 뒤, 해당 인덱스를 이용해 핵심 단어 이름과 그때의 상대 위치 값을 추출해 cluster_details라는 Dict 객체 변수에 기록하고 반환하는 함수
  centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]
  for cluster_num in range(clusters_num):
    ## 개별 군집별로 반복하며 입력

    ## 개별 군집별 정보를 담기 위해 데이터 초기화
    cluster_details[cluster_num] = {}
    cluster_details[cluster_num]['cluster'] = cluster_num

    ## cluster_centers_.argsort()[:,::-1]로 구한 인덱스 이용해 top n 피처 단어를 구함
    top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
    top_features = [feature_names[ind] for ind in top_feature_indexes]

    ## top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값을 구함
    top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()

    ## cluter_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명을 입력
    cluster_details[cluster_num]['top_features'] = top_features
    cluster_details[cluster_num]['top_features_value'] = top_feature_values
    filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']

    cluster_details[cluster_num]['filenames'] = filenames

  return cluster_details

In [42]:
# dictionary를 원소로 가지는 리스트 cluster_details를 좀 더 보기 좋게 표현하기 위해 별도의 함수 정의

def print_cluster_details(cluster_details):
  for cluster_num, cluster_detail in cluster_details.items():
    print('Cluster {0}'.format(cluster_num))
    print('Top features: ', cluster_detail['top_features'])
    print('Reviews 파일명: ', cluster_detail['filenames'][:7])
    print('===============================================')

In [43]:
# 위에서 생성한 두 함수 호출
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df, feature_names=feature_names, clusters_num=3, top_n_features=10)
print_cluster_details(cluster_details)

Cluster 0
Top features:  ['interior', 'seat', 'mileage', 'comfortable', 'gas', 'quality', 'gas mileage', 'transmission', 'button', 'car']
Reviews 파일명:  4     /content/drive/MyDrive/OpinosisDataset1
7     /content/drive/MyDrive/OpinosisDataset1
8     /content/drive/MyDrive/OpinosisDataset1
9     /content/drive/MyDrive/OpinosisDataset1
13    /content/drive/MyDrive/OpinosisDataset1
20    /content/drive/MyDrive/OpinosisDataset1
24    /content/drive/MyDrive/OpinosisDataset1
Name: filename, dtype: object
Cluster 1
Top features:  ['room', 'screen', 'battery', 'hotel', 'staff', 'location', 'battery life', 'keyboard', 'life', 'size']
Reviews 파일명:  0     /content/drive/MyDrive/OpinosisDataset1
1     /content/drive/MyDrive/OpinosisDataset1
2     /content/drive/MyDrive/OpinosisDataset1
3     /content/drive/MyDrive/OpinosisDataset1
5     /content/drive/MyDrive/OpinosisDataset1
6     /content/drive/MyDrive/OpinosisDataset1
10    /content/drive/MyDrive/OpinosisDataset1
Name: filename, dtype: object
C

Cluster #0에서는 실내 인테리어, 좌석, 연료 효율 등이 핵심 단어로 군집화됨. Cluster #1에서는 화면과 배터리 수명 등이 핵심 단어로 군집화됨. Cluster #2에서는 방과 서비스 등이 핵심 단어로 군집화됨.